# Final Model Training and External Validation

## Purpose
Train final LightGBM models on all internal training data (31 HSA complexes)
and apply them to an external validation complex to generate binding pose predictions.

## Method
1. **Hyperparameter optimization**: Optuna with GroupKFold (PDB-level) inner CV
2. **Final model training**: Train on all internal data using optimized hyperparameters
3. **Threshold calibration**: Compute best-F1 threshold from GroupKFold OOF predictions
4. **External validation**: Apply final models to external validation feature table and
   output per-pose predicted probabilities and binary predictions

## Input
- Internal training CSV with all features and labels (output of the RMSD/labeling step)
- Feature lists from `features_global/*.csv` (output of the feature selection step)
- External validation CSV with the same feature columns (output of the feature pipeline
  applied to the external complex)

## Output
- Final models: `final_model/seed_<N>/lgb_<exp>/<target>_final/{model.txt, features.csv, best_params.json, metrics.json}`
- External validation predictions: `predictions/prediction_final.csv`

## Run Order
1. Feature generation / selection notebooks (01--04)
2. LOPOCV evaluation (notebook 05, for performance assessment)
3. This notebook (final model + external validation)
4. SHAP local explanation (notebook 09, for interpretability analysis)

In [ ]:
# ================================================================
# Cell 1: Configuration
# ================================================================
# Update paths to match your local directory structure before running.

# Internal training data (output of notebook 03)
INPUT_CSV            = "/path/to/input.csv"

# External validation data (feature table for the external complex)
EXTERNAL_CSV         = "/path/to/external_features.csv"

# Feature set directory (output of notebook 04)
FEATURES_GLOBAL_DIR  = "/path/to/output/features_global"

# Output directories
MODEL_OUT_DIR        = "/path/to/output/final_model"
PREDICTION_OUT_DIR   = "/path/to/output/predictions"

SEED      = 42
SEED_TAG  = f"seed_{SEED}"

TARGETS = ["label_2p5", "label_5p0"]
PDB_COL = "pdb_id"

EXPERIMENTS = [
    {"name": "ds"},
    {"name": "ds_desc"},
    {"name": "ds_md"},
    {"name": "ds_desc_md"},
]

# Optuna hyperparameter search
N_TRIALS     = 400   # number of Optuna trials
INNER_SPLITS = 5     # GroupKFold splits for inner CV
EARLY_STOP   = 20    # early stopping rounds

In [ ]:
# ================================================================
# Cell 2: Libraries
# ================================================================
import os, json, warnings, time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve
)

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ================================================================
# Cell 3: Utilities
# ================================================================
def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def scale_pos_weight_from_y(y: np.ndarray) -> float:
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    return float(neg / max(pos, 1))

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> Tuple[float, float]:
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    thresholds = np.r_[0.0, thr, 1.0]
    f1s = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
    i = int(np.argmax(f1s))
    return float(thresholds[i]), float(f1s[i])

def lgb_base_params() -> Dict:
    return {
        "device_type": "cpu",
        "num_threads": -1,      # use all available threads (-1 = auto)
        "tree_learner": "serial",
        "objective": "binary",
        "verbosity": -1,
        "feature_pre_filter": False,
        "force_row_wise": True,
        "deterministic": True,
        "seed": SEED,
    }

def load_global_features(exp_name: str) -> List[str]:
    feat_path = Path(FEATURES_GLOBAL_DIR) / f"{exp_name}.csv"
    if not feat_path.exists():
        raise FileNotFoundError(
            f"features_global not found: {feat_path}\n"
            "Please run notebook 04 (feature selection) first."
        )
    feats = pd.read_csv(feat_path)["feature"].astype(str).tolist()
    if len(feats) == 0:
        raise ValueError(f"features_global is empty: {feat_path}")
    return feats

# Custom eval metric: Average Precision (AP)
def feval_ap(y_pred: np.ndarray, data: lgb.Dataset):
    y_true = data.get_label()
    if len(np.unique(y_true)) < 2:
        return "ap", 0.0, True
    return "ap", float(average_precision_score(y_true, y_pred)), True

In [ ]:
# ================================================================
# Cell 4: Data loading
# ================================================================
df_raw = pd.read_csv(INPUT_CSV)
df_raw = df_raw[df_raw["ligand_id"].astype(str).str.startswith("pose")].copy()
print(f"[Data] Internal training rows (pose only): {len(df_raw)}")

In [ ]:
# ================================================================
# Cell 5: Optuna objective and OOF threshold calibration
# ================================================================
def make_objective(X, y, groups, inner_splits, early_stopping_rounds=EARLY_STOP):
    def objective(trial: optuna.Trial) -> float:
        params = {
            **lgb_base_params(),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth":         trial.suggest_int("max_depth", 3, 10),
            "num_leaves":        trial.suggest_int("num_leaves", 15, 255),
            "feature_fraction":  trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq":      trial.suggest_int("bagging_freq", 0, 5),
            "min_data_in_leaf":  trial.suggest_int("min_data_in_leaf", 10, 200),
            "lambda_l1":         trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2":         trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 2.0),
            "max_bin":           trial.suggest_int("max_bin", 128, 512),
            "metric": "None",
        }
        n_estimators = trial.suggest_int("n_estimators", 100, 1200)

        gkf = GroupKFold(n_splits=min(inner_splits, len(np.unique(groups))))
        scores, best_iters = [], []
        step = 0

        for tr_idx, va_idx in gkf.split(X, y, groups):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            if len(np.unique(y_va)) < 2:
                continue

            params["scale_pos_weight"] = scale_pos_weight_from_y(y_tr)
            dtr = lgb.Dataset(X_tr, label=y_tr, free_raw_data=True)
            dva = lgb.Dataset(X_va, label=y_va, reference=dtr, free_raw_data=True)

            bst = lgb.train(
                params, dtr,
                num_boost_round=n_estimators,
                valid_sets=[dva], valid_names=["valid"],
                feval=feval_ap,
                callbacks=[
                    lgb.early_stopping(early_stopping_rounds, verbose=False),
                    lgb.log_evaluation(period=0),
                ],
            )
            best_iters.append(int(bst.best_iteration))
            y_va_prob = bst.predict(X_va, num_iteration=bst.best_iteration)
            scores.append(float(average_precision_score(y_va, y_va_prob)))

            step += 1
            trial.report(float(np.mean(scores)), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if len(scores) == 0:
            return float(np.mean(y))

        trial.set_user_attr("mean_best_iteration",
                            int(np.mean(best_iters)) if best_iters else n_estimators)
        return float(np.mean(scores))

    return objective

def compute_oof_threshold(X, y, groups, booster_params, num_boost_round,
                          early_stopping_rounds=EARLY_STOP):
    """Compute best-F1 threshold via GroupKFold OOF predictions."""
    gkf = GroupKFold(n_splits=min(5, len(np.unique(groups))))
    oof = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in gkf.split(X, y, groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        if len(np.unique(y_va)) < 2:
            oof[va_idx] = 0.0
            continue

        p = dict(booster_params)
        p["scale_pos_weight"] = scale_pos_weight_from_y(y_tr)
        p["metric"] = "None"

        dtr = lgb.Dataset(X_tr, label=y_tr, free_raw_data=True)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr, free_raw_data=True)

        bst = lgb.train(
            p, dtr,
            num_boost_round=num_boost_round,
            valid_sets=[dva], valid_names=["valid"],
            feval=feval_ap,
            callbacks=[
                lgb.early_stopping(early_stopping_rounds, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )
        oof[va_idx] = bst.predict(X_va, num_iteration=bst.best_iteration)

    thr, f1 = best_f1_threshold(y, oof)
    return thr, f1, oof

In [ ]:
# ================================================================
# Cell 6: Train final models
# ================================================================
start_t = time.time()
thresholds = {}  # {(exp_name, target): threshold}

for exp in EXPERIMENTS:
    exp_name = exp["name"]
    print(f"\n=== Finalizing: {exp_name} ===")

    feats = load_global_features(exp_name)
    missing = [c for c in feats if c not in df_raw.columns]
    if missing:
        raise KeyError(f"Missing columns for {exp_name}: {missing[:10]}")

    X_all = df_raw[feats].copy()

    for target in TARGETS:
        print(f"\n----- Target: {target} -----")
        y_all = df_raw[target].astype(int).values
        groups_all = df_raw[PDB_COL].astype(str).values

        # Optuna hyperparameter search
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED),
            pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
        )
        objective = make_objective(X_all, y_all, groups_all, INNER_SPLITS, EARLY_STOP)
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

        best = study.best_trial.params
        proposed_n = int(best.get("n_estimators", 500))
        mean_best_iter = int(study.best_trial.user_attrs.get("mean_best_iteration", proposed_n))
        best_n = max(10, min(mean_best_iter, proposed_n))

        params = {
            **lgb_base_params(),
            **{k: v for k, v in best.items() if k != "n_estimators"},
            "scale_pos_weight": scale_pos_weight_from_y(y_all),
        }

        # Train final model on all internal data
        bst = lgb.train(params, lgb.Dataset(X_all, label=y_all), num_boost_round=best_n)

        # OOF-based best-F1 threshold
        best_thr, best_f1, oof_prob = compute_oof_threshold(
            X_all, y_all, groups_all, params, best_n, EARLY_STOP
        )
        thresholds[(exp_name, target)] = best_thr

        # Save model artifacts
        base_dir = Path(MODEL_OUT_DIR) / SEED_TAG / f"lgb_{exp_name}" / f"{target}_final"
        base_dir.mkdir(parents=True, exist_ok=True)

        bst.save_model(str(base_dir / "model.txt"), num_iteration=best_n)
        pd.Series(X_all.columns, name="feature").to_csv(base_dir / "features.csv", index=False)

        best_params_save = best.copy()
        best_params_save["n_estimators_used"] = int(best_n)
        best_params_save["n_estimators_proposed"] = int(proposed_n)
        best_params_save["optuna_best_value(ap_mean_cv)"] = float(study.best_value)
        with open(base_dir / "best_params.json", "w") as f:
            json.dump(best_params_save, f, indent=2)

        metrics = {
            "n_samples": int(len(X_all)),
            "n_features": int(X_all.shape[1]),
            "n_groups": int(len(np.unique(groups_all))),
            "n_estimators_used": int(best_n),
            "oof_auc": float(roc_auc_score(y_all, oof_prob)) if len(np.unique(y_all)) == 2 else float("nan"),
            "oof_pr_auc": float(average_precision_score(y_all, oof_prob)),
            "bestF1_threshold": float(best_thr),
            "bestF1_oof": float(best_f1),
        }
        with open(base_dir / "metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

        print(f"[OK] saved: {base_dir}")
        print(f"     best-F1 thr={best_thr:.4f} (OOF F1={best_f1:.4f}) "
              f"OOF AUC={metrics['oof_auc']:.4f}  OOF PR-AUC={metrics['oof_pr_auc']:.4f}")

elapsed = time.time() - start_t
print(f"\nAll final models done in {elapsed:.1f} sec")

## External Validation

Apply the trained final models to the external validation complex and generate
per-pose predictions with calibrated thresholds from the OOF step above.

In [ ]:
# ================================================================
# Cell 7: External validation — predict on new complex
# ================================================================
df_ext = pd.read_csv(EXTERNAL_CSV)
print(f"[External] rows: {len(df_ext)}")

pred_out_dir = Path(PREDICTION_OUT_DIR) / SEED_TAG
pred_out_dir.mkdir(parents=True, exist_ok=True)

# Collect predictions from all experiment x target combinations
pred_df = df_ext[["pdb_id", "ligand_id"]].copy()
if "smiles" in df_ext.columns:
    pred_df["smiles"] = df_ext["smiles"]
if "rmsd_gninaA" in df_ext.columns:
    pred_df["rmsd"] = df_ext["rmsd_gninaA"]

for exp in EXPERIMENTS:
    exp_name = exp["name"]
    feats = load_global_features(exp_name)
    missing = [c for c in feats if c not in df_ext.columns]
    if missing:
        print(f"[WARN] {exp_name}: {len(missing)} feature(s) missing — skipping")
        continue

    X_ext = df_ext[feats].copy()

    for target in TARGETS:
        model_dir = Path(MODEL_OUT_DIR) / SEED_TAG / f"lgb_{exp_name}" / f"{target}_final"
        model_path = model_dir / "model.txt"
        if not model_path.exists():
            print(f"[WARN] Model not found: {model_path} — skipping")
            continue

        bst = lgb.Booster(model_file=str(model_path))
        prob = bst.predict(X_ext.values)

        thr = thresholds.get((exp_name, target), 0.5)
        pred = (prob >= thr).astype(int)

        col_prob = f"{exp_name}__prob_{target}"
        col_pred = f"{exp_name}__pred_{target}"
        pred_df[col_prob] = np.round(prob, 6)
        pred_df[col_pred] = pred

out_csv = pred_out_dir / "prediction_final.csv"
pred_df.to_csv(out_csv, index=False)
print(f"\n[OK] Predictions saved: {out_csv}")
print(f"     Shape: {pred_df.shape}")
print(pred_df.to_string(index=False))